# Вычисление конверсии

Например, рассмотрим продуктовую метрику "конверсия из показа карточки контента в просмотр":  
* пользователь заходит на главную страницу ivi
* рекомендательная система показывает постеры нескольких карточек контента
* если рекомендательная система угадала вкусы пользователя - начинается просмотр контента. Если выдача рекомендаций пользователю не подходит - событие "просмотр контента" не возникает

![main_page_ivi](img/main_page_ivi.png)

# A/B тестирование (Оценивание гипотез)

## Таблица ошибок для A/B теста

Гипотеза: значения p-value определяют должны ли мы отклонить нулевую гипотезу.

|           | Accept H₀ | Reject H₀ |
|-----------|-----------|-----------|
| H₀ is true | ✓         | I error (FP) |
| H₀ is false | II error (FN) | TN |

**Пояснения:**
- **Нулевая гипотеза**: Эффекта нет (группы не различаются)
- **I error (FP)**: Ложноположительное - отклонили нулевую гипотезу, когда она верна (сказали что эффект есть, хотя его нет)
- **II error (FN)**: Ложноотрицательное - приняли нулевую гипотезу, когда она ложна (не заметили реальный эффект)
- **Распределение p-value**: Альтернативное H₀/H₁ при условии V-правильности нулевой гипотезы

## Статистическая мощность теста

**Статистическая мощность** - вероятность заврата ЕСЛИ он есть. Точнее: вероятность 80% означает, что мы правильно обнаружим различие в 8 случаях из 10, когда эффект действительно есть (т.е. данные минимального обнаруженного эффекта).

### Формула для расчета размера выборки

```
α - significance level
β - statistical power

         (z₁₋α + z₁₋β)² σ²
n = ─────────────────────────
              Δ²
```

Где:
- **Δ² = absolute difference** - абсолютная разница
- **n = sample size** - размер выборки
- **Δ = (θ₂ - θ₁)** → mean (μ₂ - μ₁)² / proportion (p₂ - p₁)

### Пример расчета

**Дано:**
- MDE = 20% (Minimum Detectable Effect)
- CTR_A = 15%

**Решение:**
```
CTR_B = CTR_A(1 + MDE/100) = 0.18

Δ² = (0.18 - 0.15)² = 0.0009
```

**Размер эффекта** = отношение на Δ²

```
n ≈ 160/Δ²
```

## Ключевые выводы

1. **Статистическая мощность (80%)** означает, что мы правильно обнаружим эффект в 8 случаях из 10, когда он действительно существует.

2. **Размер выборки обратно пропорционален Δ²**: чем меньше эффект, который мы хотим детектировать, тем больше нужна выборка.

3. **Типы ошибок:**
   - **Type I Error (α)**: Ложная тревога - решили, что эффект есть, когда его нет
   - **Type II Error (β)**: Пропуск эффекта - не заметили реальный эффект

4. **Формула n ≈ 160/Δ²** - это упрощенная формула для быстрой оценки при α=0.05 и power=0.80.

5. Чтобы увеличить мощность теста надо увеличить выборку (будем долго ждать) или уменьшить дисперсию. MDE - чем чувствительнее тесты тем лучше улавливается эффект, но сами вычисления обычно дороже.

Для категориальных переенных можноиспользовать chi-square test

### Пример


Вычислим эту метрику с помощью SQL на основании данных событийной аналитики
* берём только события с платформ `xboxOne` и `Windows 10`
* тип события `page_impression` (показ контента) и `click` (клик по постеру)
* валидация событий: просмотры, у которых отсутствует `impression` из выборки исключаем
* считаем конверсию по датам в разбивке по платформам

Цель исследования - понять, есть ли отличия в конверсии между просмотрами на платформах `xboxOne` и `Windows 10`

# Расчёт конверсии

Если всё сделали правильно - код ниже выведет значение конверсии по дням для двух самых популярных `subsite_title` в виде таблички

In [4]:
%load_ext autoreload
%autoreload 2

import sys
import os

import numpy as np

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)  # гарантируем воспроизводимость

run_env = os.getenv('RUN_ENV', 'COLLAB')
if run_env == 'COLLAB':
  from google.colab import drive
  ROOT_DIR = '/content/drive'
  drive.mount(ROOT_DIR)
  print('Google drive connected')
  DRIVE_DATA_DIR = 'ml_course_data'
  root_data_dir = os.path.join(ROOT_DIR, 'MyDrive', DRIVE_DATA_DIR)
  sys.path.append(os.path.join(ROOT_DIR, 'MyDrive', 'src'))
else:
  root_data_dir = os.getenv('DATA_DIR', '/srv/data')

if not os.path.exists(root_data_dir):
  raise RuntimeError('Отсутствует директория с данными')
else:
  print('Содержимое директории %s: %s' % (root_data_dir, os.listdir(root_data_dir)[:10]))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Содержимое директории /Users/adzhumurat/PycharmProjects/ai_product_engineer/data: ['client_segmentation.csv', 'messages.db', 'labeled_data_corpus.csv', 'chroma', 'content_description.csv', 'nltk_data', 'content_catalog.zip', 'user_item_views.zip', 'sms+spam+collection.zip', 'pipelines-data']


In [5]:
import pandas as pd

conversion_df = pd.read_json(os.path.join(root_data_dir, 'conversions.jsonl'), lines=True)

conversion_df.head()

,date,subsite_title,conversion
0,2019-06-01,Windows 10,0.197802
1,2019-06-01,xboxOne,0.200704
2,2019-06-02,Windows 10,0.287532
3,2019-06-02,xboxOne,0.270677
4,2019-06-03,Windows 10,0.439331


# Задание 4

Постройте график следующего вида
* значение конверсии по дням
* каждый `subsite_title` отдельной линии

Должно получится как-то так

![conversions](img/conversions.png)

Готово! Поздравляю с успешным выполнением домашки